# ⛰ Core Lab 2 — How a Neuron Learns
**A Billion Small Steps · Hardware for AI · Step 1 · Book sections 1.3–1.4 · Foothold lessons 1.3–1.4**

Read the book sections first, then run this notebook from top to bottom. Runs on the **CPU** on purpose: the default runtime is fine.

**What you should see:** the bias moving the cut-off; the nudge-test slope matching the formula; a learning rate that is too large diverging; backprop slopes matching the nudge test.

| Part | What you do | Book |
|---|---|---|
| A | Build a neuron with a loop, then with `np.dot`; see why it needs a bias | 1.3 |
| B | The nudge test, gradient descent from scratch, and backpropagation checked against the nudge test | 1.4 |

⏱️ About 2 hours. 📘 Textbook: D2L §3.1–3.4, §5.1. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **No GPU needed for this lab.** The default CPU runtime is fine — that is deliberate: you will compare against the GPU later.
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import numpy as np                 # NumPy: arrays of numbers and fast maths on them ("np" is its short name)
import matplotlib.pyplot as plt    # Matplotlib: draws charts and pictures ("plt" is its short name)
import time                        # time: we use its stopwatch, time.perf_counter()

# Print arrays neatly: wide lines, 3 decimal places, no 1e-05 style numbers
np.set_printoptions(linewidth=150, precision=3, suppress=True)
print("Ready! NumPy version:", np.__version__)   # show that everything loaded, and NumPy's version

## Part A · The Neuron  *(Section 1.3)*

A neuron: multiply each input by its weight, add them up, add the bias, then decide.

$$z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b$$

First with a plain Python loop — exactly what the hardware does — then with NumPy's `np.dot`.

In [ ]:
# Nila's question tonight: go to the IPL match (YES) or stay home and study (NO)?
# The three inputs, written as 1 (yes) or 0 (no): sunny? yes · exam tomorrow? yes · friends going? no
x = np.array([1, 1, 0])
# How much each input matters: sunny pulls towards GO (+2), an exam pulls hard towards STAY (-4), friends pull a little towards GO (+1.5)
w = np.array([2.0, -4.0, 1.5])
b = -1.0                         # the bias: one fixed number added at the end (here it leans slightly towards STAY)

# The loop way: one multiply-add (MAC) at a time, exactly the way the hardware does it
z_loop = b                       # start the running total at the bias
for i in range(len(x)):          # visit the inputs one by one: i = 0, then 1, then 2
    z_loop += w[i] * x[i]        # multiply this input by its weight and add the result to the running total

# The NumPy way: np.dot multiplies the pairs and adds them all up in one call; then we add the bias
z_dot = np.dot(w, x) + b

print("z (loop)   =", z_loop)    # show the total the loop worked out
print("z (np.dot) =", z_dot)     # show the total np.dot worked out: it should be the same number
# Step activation: if z is 0 or more the answer is YES (go to the match), otherwise NO (stay home)
print("Decision:", "GO to the match" if z_dot >= 0 else "STAY home")

### Why does a neuron need a bias?
Back to the 12 students from Core Lab 1, Part A, with **one** input: study hours. The neuron computes `z = w × hours + b` and says PASS when `z ≥ 0`.
It switches from FAIL to PASS exactly where `z = 0`, that is at **hours = −b ÷ w**. That switching point is the neuron's **cut-off**.

Run it twice: first with **no bias** (`b = 0`), then with `b = −3`.

In [ ]:
# The 12 students from Core Lab 1, Part A, again. New names (hrs, res) because Part B uses "hours" for different data.
hrs = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.0, 3.5, 4.0, 4.5, 5.5, 6.0])   # study hours per day
res = np.array([0,   0,   0,   0,   0,   0,   1,   1,   1,   1,   1,   1  ])   # real result: 1 = passed, 0 = failed

def one_input_neuron(w, b):               # a neuron with ONE input (hours); w and b are its two knobs
    z = w * hrs + b                       # weighted sum for all 12 students at once: weight × hours + bias
    pred = (z >= 0).astype(int)           # step activation: 1 (PASS) where z is 0 or more, else 0 (FAIL)
    correct = (pred == res).sum()         # compare with the real results and count how many match
    cut = -b / w + 0.0                    # the cut-off is where z = 0, i.e. hours = -b ÷ w (+ 0.0 stops it printing "-0.0")
    # print the two knobs, the cut-off they produce, and the score out of 12
    print(f"w = {w}, b = {b:5}:  cut-off = -b / w = {cut:4.1f} hours   correct = {correct} / 12")

one_input_neuron(w=1.0, b=0.0)    # no bias: z = hours, which is >= 0 for EVERY student, so everyone "passes"
one_input_neuron(w=3.0, b=0.0)    # a bigger weight doesn't help: the cut-off is still stuck at 0 hours
one_input_neuron(w=1.0, b=-3.0)   # with a bias: "start 3 hours in debt", pass once the debt is paid off

Without a bias, `z = w × hours` is 0 only at 0 hours, so the cut-off can never move, whatever `w` is. **The weight says how much each hour counts; the bias says where the pass mark is.** With `b = −3` the neuron reproduces the rule the machine learned in Part A (pass at 3 hours, 11 / 12).

### ✏️ YOUR TURN (A1)
1. Which `b` puts the cut-off at 4 hours when `w = 1`? Which `b` does it when `w = 2`? Check both with `one_input_neuron`.
2. Try `w = -1.0, b = 3.0`. Where is the cut-off now, and which side of it says PASS?

*Your answers:*

**Activation functions** bend the output. Plot the three from the lesson:

In [ ]:
z = np.linspace(-5, 5, 200)          # 200 evenly spaced z values from -5 to 5 (the bottom axis of the charts)
step    = (z >= 0).astype(float)     # step: 1 if z >= 0, else 0 (a hard YES/NO)
sigmoid = 1 / (1 + np.exp(-z))       # sigmoid: squashes any z into the range 0..1 (a smooth "how sure")
relu    = np.maximum(0, z)           # ReLU: a negative z becomes 0, a positive z stays the same

plt.figure(figsize=(9, 2.8))         # one wide drawing area (9 × 2.8 inches) to hold three small charts
# Go through the three activations: k counts 0, 1, 2 and (name, y) is each one's name and output values
for k, (name, y) in enumerate([("Step", step), ("Sigmoid", sigmoid), ("ReLU", relu)]):
    plt.subplot(1, 3, k + 1)         # split the area into 1 row × 3 boxes and draw in box number k+1
    plt.plot(z, y)                   # draw the curve: z along the bottom, the activation's output going up
    plt.title(name)                  # write the activation's name above its box
    plt.grid(alpha=0.3)              # add faint grid lines so values are easier to read
plt.tight_layout()                   # space the three charts so they don't overlap
plt.show()                           # display the finished picture

Now the six students from the lesson, whose results depend on **both** hours and attendance.

In [ ]:
students = ["KAVYA", "RAVI", "SURESH", "GEETHA", "IMRAN", "POOJA"]          # the six names, used only for printing
X = np.array([[4, 40], [2, 95], [5, 90], [1, 60], [3.5, 85], [2.5, 50]])   # one row per student: hours, attendance %
y = np.array([0, 1, 1, 0, 1, 0])                                           # the real results: 1 = pass, 0 = fail

w = np.array([1.0, 0.05])     # ✏️ weights: each hour counts 1.0, each attendance % counts 0.05
b = -6.5                      # ✏️ bias: sets where the pass mark sits

z = X @ w + b                 # @ = matrix multiply: one weighted sum per student, all six in one go
pred = (z >= 0).astype(int)   # step activation: 1 (PASS) where z >= 0, else 0 (FAIL)
for s, zi, p, t in zip(students, z, pred, y):    # walk through name, z, prediction and real result together
    # print the student's z, the neuron's decision, and a tick if it matches what really happened
    print(f"{s:7s} z = {zi:6.2f} → {'PASS' if p else 'FAIL'}  {'✓' if p == t else '✗'}")
print("Correct:", (pred == y).sum(), "/ 6")   # count how many predictions match the real results

### ✏️ YOUR TURN (A2)
1. Set `w = np.array([1.0, 0.0])` (ignore attendance). How many are right now? Which students go wrong?
2. A neuron reading a 28 × 28 image has how many weights? How many multiplications per image?

*Your answers:*

## Part B · Gradient Descent  *(Section 1.4)*

Five of Nila's classmates studied 1, 2, 3, 4 and 5 hours and scored 30, 45, 55, 70 and 80. Model: `marks = w × hours + 20`. It has **one knob**, `w`. We want the `w` that makes the **mean squared error** (the loss) smallest.

**Step 1 · Score some guesses.** Compute the loss for a few settings of the knob.

In [ ]:
hours = np.array([1, 2, 3, 4, 5])          # 5 classmates' study hours
marks = np.array([30, 45, 55, 70, 80])     # the marks they really got

def loss_fn(w):                            # the score for one setting of the knob w (lower = better)
    pred = w * hours + 20                  # the model's predicted marks for all 5 students
    return np.mean((marks - pred) ** 2)    # error = real − predicted; square each one, then average them (MSE)

for w in [8, 10, 12, 14, 16]:              # five guesses for the knob
    print(f"w = {w:2d}   loss = {loss_fn(w):6.1f}")   # print each guess next to its loss

Down, then up again: the loss makes a **valley**, with its bottom near 12. With one knob we could try every setting. The network in Core Lab 3 has 101,770 knobs, so trying them all is hopeless. We need to know **which way is downhill** from where we stand.

**Step 2 · The nudge test.** Move the knob a tiny bit (+0.1) and see how the loss changes. Change ÷ nudge = the **slope** (the gradient).

In [ ]:
def nudge_slope(w, nudge=0.1):             # the nudge test at the point w
    # (loss a tiny step to the right − loss here) ÷ size of the step = the slope
    return (loss_fn(w + nudge) - loss_fn(w)) / nudge

def grad_fn(w):                            # the EXACT slope at w, from calculus (for this data it equals 22*w - 266)
    pred = w * hours + 20                  # the predictions for this w
    return np.mean(-2 * hours * (marks - pred))   # the calculus formula for the slope of the MSE

for w in [8, 16, 12.09]:                   # three places: left of the bottom, right of it, and at the bottom
    s = nudge_slope(w)                     # feel the slope where we stand
    # negative slope: going right lowers the loss; positive: going right raises it; near 0: flat ground
    advice = "increase w" if s < -5 else "decrease w" if s > 5 else "almost flat: at the bottom"
    # print the loss before and after the nudge, the nudge slope, the exact slope, and which way to move
    print(f"w = {w:5}:  loss {loss_fn(w):7.2f} -> {loss_fn(w + 0.1):7.2f}   nudge slope = {s:6.1f}   exact = {grad_fn(w):6.1f}   -> {advice}")

A negative slope means "going right makes the loss smaller", so increase `w`. A positive slope means decrease `w`. One rule covers both: **`new w = w − learning_rate × slope`**. The minus sign *is* "go downhill".

**Step 3 · Why a step neuron can't learn.** Try the nudge test on the pass/fail neuron from Part A (`w = 1`), nudging the **bias** by 0.01. With the step activation the loss is "fraction of students wrong"; with sigmoid it is the average squared gap between the output (0 to 1) and the real answer.

In [ ]:
def step_loss(b):                                    # the loss of the STEP neuron for bias b (weight fixed at 1)
    pred = (1.0 * hrs + b >= 0).astype(int)          # hard PASS (1) / FAIL (0) for each of the 12 students
    return np.mean(pred != res)                      # the fraction of students it gets wrong

def sigmoid_loss(b):                                 # the loss of the same neuron with SIGMOID instead of step
    out = 1 / (1 + np.exp(-(1.0 * hrs + b)))         # a smooth "how sure" between 0 and 1 for each student
    return np.mean((out - res) ** 2)                 # the average squared gap from the real answer

b = -2.6                                             # where we stand: a bias of -2.6
for name, f in [("step", step_loss), ("sigmoid", sigmoid_loss)]:   # do the nudge test on both neurons
    s = (f(b + 0.01) - f(b)) / 0.01                  # nudge b by 0.01 and measure the slope
    print(f"{name:8s} loss {f(b):.4f} -> {f(b + 0.01):.4f}   slope = {s:.3f}")   # loss before, after, and the slope

The step loss doesn't move at all: a tiny nudge flips no PASS/FAIL answer, so the slope is 0 and gradient descent has **no direction**. Sigmoid always changes a little, so there is always a slope to follow. That is why real networks use smooth activations (sigmoid, ReLU).

**Step 4 · Walk downhill.** Repeat "feel the slope, take a step" from `w = 0`:

In [ ]:
# Gradient descent: start at w, then take "epochs" steps, each one moving w by lr × slope against the slope
def train(lr, epochs=15, w=0.0, verbose=False):
    history = []                               # an empty list that will remember the loss at every step (for plotting)
    for epoch in range(epochs):                # repeat once for every step (epoch)
        history.append(loss_fn(w))             # write down how wrong the model is right now
        if verbose:                            # only when we asked for printing...
            # ...show the step number, the knob, the loss and the slope
            print(f"epoch {epoch:2d}  w = {w:8.3f}  loss = {loss_fn(w):10.2f}  grad = {grad_fn(w):9.2f}")
        w = w - lr * grad_fn(w)                # THE step: move w a little against the slope (downhill)
    return w, history                          # hand back the final knob and the list of losses

# Learning rate 0.02, 8 steps, print every one ("_" means "I don't need the loss list here")
w_final, _ = train(lr=0.02, epochs=8, verbose=True)

Compare learning rates — the balancing act from the lesson:

In [ ]:
plt.figure(figsize=(7, 3.5))                          # a fresh drawing area, 7 × 3.5 inches
for lr in [0.005, 0.02, 0.08, 0.1]:                   # four learning rates to compare
    w_end, hist = train(lr, epochs=15)                # train with this one; keep the final w and the list of losses
    plt.plot(hist, marker="o", label=f"lr = {lr}  (final w = {w_end:.2f})")   # draw one line per learning rate
plt.yscale("log")                                     # log scale, so huge and tiny losses both fit on one chart
plt.xlabel("epoch"); plt.ylabel("loss (log scale)")   # label the bottom axis and the side axis
plt.legend(); plt.grid(alpha=0.3)                     # show the colour key and faint grid lines
plt.title("Too small crawls · just right settles · too big zig-zags or explodes")   # the chart's heading
plt.show()                                            # display the chart

### ✏️ YOUR TURN (B1)
1. Which learning rate reached w ≈ 12.09 fastest? What happened at 0.1?
2. Change the starting point to `w=30.0` in `train(...)`. Does it still find the bottom?

*Your answers:*

### B2 · Check backpropagation against the nudge test
The two-weight chain from the book's "Backpropagation by hand" (Section 1.4): x → (× w1) → h → (× w2) → y.
The **backward pass** starts at the loss and passes the slope back one link at a time. The **nudge test** checks each slope the slow way.

In [ ]:
# one example: the input and the answer we want
x, t = 2.0, 5.0
# the two knobs, in a chain: x → (times w1) → h → (times w2) → y
w1, w2 = 0.5, 3.0

# the whole chain, forward: input → h → y → loss
def chain_loss(w1, w2):
    # first multiply, then second multiply, then the squared error
    return (w2 * (w1 * x) - t) ** 2

# forward pass: keep h, because the backward pass needs it
h = w1 * x
# the prediction
y = w2 * h
# the loss for this one example
L = (y - t) ** 2

# backward pass: start at the loss and pass the slope back, one link at a time
# slope of the loss with respect to y
dL_dy = 2 * (y - t)
# slope for w2 = slope arriving at y × what w2 was multiplied by (h)
dL_dw2 = dL_dy * h
# slope passed back to h = slope at y × the weight on that link (w2)
dL_dh = dL_dy * w2
# slope for w1 = slope arriving at h × what w1 was multiplied by (x)
dL_dw1 = dL_dh * x

# the nudge test, to check: a tiny step for each knob in turn
nudge = 0.001
# how much the loss changes per unit of nudge on w1
nudge_w1 = (chain_loss(w1 + nudge, w2) - L) / nudge
# the same for w2
nudge_w2 = (chain_loss(w1, w2 + nudge) - L) / nudge

# print the forward numbers
print(f"forward:  h = {h}   y = {y}   loss = {L}")
# print the two methods side by side
print(f"w1: backprop slope = {dL_dw1:6.1f}   nudge slope = {nudge_w1:8.3f}")
# and the same for w2
print(f"w2: backprop slope = {dL_dw2:6.1f}   nudge slope = {nudge_w2:8.3f}")

# one gradient-descent step with learning rate 0.01, using the backprop slopes
w1, w2 = w1 - 0.01 * dL_dw1, w2 - 0.01 * dL_dw2
# the new knobs and the new loss
print(f"after one step: w1 = {w1:.2f}   w2 = {w2:.2f}   loss = {chain_loss(w1, w2):.4f}")

Both methods agree (the nudge slopes differ only in the third decimal, because a nudge of 0.001 is small but not zero). Backpropagation gets **every** slope from one backward pass; the nudge test needs one extra forward pass **per weight**. With a million weights, that is the difference between one pass and a million.

### ✏️ YOUR TURN (B2)
1. Change `x` to 3. Work out the four backward slopes (dL_dy, dL_dw2, dL_dh, dL_dw1) **by hand** first, then run the cell to check.
2. Why does the backward pass need `h`, the number kept from the forward pass? (Section 2.2 uses this to explain why training needs more memory than inference.)

*Your answers:*

---
## 🎓 Wrap-up
- A neuron = weighted sum + bias, then an activation; the bias moves the cut-off.
- Gradient descent: measure the slope of the loss, take a small step downhill, repeat. Too big a step diverges.
- Backpropagation gives every weight's slope in one backward pass, and it agrees with the nudge test.
- **Next — Core Lab 3: Train a Real Network.** Layers as matrix multiplies, and a network that reads handwriting.